In [1]:
## Setup — resolve project paths"

import sys
from pathlib import Path

def find_src_dir(start: Path = None) -> Path:
    """Walk upward from the current directory until a 'src' folder is found."""
    current = start or Path.cwd()
    for _ in range(5):
        candidate = current / "src"
        if candidate.exists():
            return candidate
        current = current.parent
    raise FileNotFoundError("Could not locate a 'src' folder above the current directory.")

SRC_DIR = find_src_dir()
sys.path.append(str(SRC_DIR))
from paths import PROJECT_ROOT, ZIP_DIR, DATA_DIR, EXTRACTED_DIR, PARQUET_DIR, IBES_DIR

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL


In [2]:
# Rebuild pipeline (optional - only needed for first time setup)

from extract_zips import run_extraction
from ingest_cboe import run_ingestion

if not any(PARQUET_DIR.glob("*.parquet")):
    run_extraction()
    run_ingestion(raw_dir=str(EXTRACTED_DIR), out_dir=str(PARQUET_DIR))
else:
    print("Parquet files already exist -- skipping. Delete data/cboe_parquet/ first before rebuilding.")

Found 2862 per-day zip files to convert

  100/2862 done (0.0 min elapsed, ~1 min remaining)
  200/2862 done (0.0 min elapsed, ~1 min remaining)
  300/2862 done (0.1 min elapsed, ~1 min remaining)
  400/2862 done (0.1 min elapsed, ~1 min remaining)
  500/2862 done (0.1 min elapsed, ~1 min remaining)
  600/2862 done (0.2 min elapsed, ~1 min remaining)
  700/2862 done (0.2 min elapsed, ~1 min remaining)
  800/2862 done (0.2 min elapsed, ~1 min remaining)
  900/2862 done (0.3 min elapsed, ~1 min remaining)
  1000/2862 done (0.3 min elapsed, ~1 min remaining)
  1100/2862 done (0.3 min elapsed, ~1 min remaining)
  1200/2862 done (0.4 min elapsed, ~1 min remaining)
  1300/2862 done (0.4 min elapsed, ~0 min remaining)
  1400/2862 done (0.4 min elapsed, ~0 min remaining)
  1500/2862 done (0.5 min elapsed, ~0 min remaining)
  1600/2862 done (0.5 min elapsed, ~0 min remaining)
  1700/2862 done (0.5 min elapsed, ~0 min remaining)
  1800/2862 done (0.6 min elapsed, ~0 min remaining)
  1900/2862 do

In [3]:
# Verification

from verify_setup import main
main()

=== Yearly Parquet row counts ===
  cboe_openclose_2011.parquet: 7,587,825 rows
  cboe_openclose_2012.parquet: 7,429,806 rows
  cboe_openclose_2013.parquet: 7,849,786 rows
  cboe_openclose_2014.parquet: 10,045,109 rows
  cboe_openclose_2015.parquet: 8,900,411 rows
  cboe_openclose_2016.parquet: 9,324,066 rows
  cboe_openclose_2017.parquet: 10,834,820 rows
  cboe_openclose_2018.parquet: 13,921,941 rows
  cboe_openclose_2019.parquet: 14,774,622 rows
  cboe_openclose_2020.parquet: 20,826,043 rows
  cboe_openclose_2021.parquet: 24,336,763 rows
  cboe_openclose_2022.parquet: 9,047,229 rows
  Total: 144,878,421 rows across 12 file(s)

=== Disk usage ===
  Raw zip archives: 4.19 GB -- C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL\CBOE_Data_2011_2022
  Parquet output: 3.95 GB -- C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL\data\cboe_parquet
  IBES data: 0.00 GB -- C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL\data\ibes_quarterly_report
  Free space: 142.2 GB

=== Spot check against a fresh

In [1]:
import polars as pl
df = pl.read_parquet(r"C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL\data\ibes_quarterly_report\ibes_clean.parquet")

print("Unique companies:", df["CNAME"].n_unique())
print("\nMEASURE values:")
print(df["MEASURE"].value_counts())
print("\nFISCALP values:")
print(df["FISCALP"].value_counts())
print("\nSTATPERS range:", df["STATPERS"].min(), "to", df["STATPERS"].max())
print("ANNDATS_ACT range:", df["ANNDATS_ACT"].min(), "to", df["ANNDATS_ACT"].max())

Unique companies: 15269

MEASURE values:
shape: (1, 2)
┌─────────┬─────────┐
│ MEASURE ┆ count   │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ EPS     ┆ 4932134 │
└─────────┴─────────┘

FISCALP values:
shape: (2, 2)
┌─────────┬─────────┐
│ FISCALP ┆ count   │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ QTR     ┆ 3137755 │
│ ANN     ┆ 1794379 │
└─────────┴─────────┘

STATPERS range: None to None
ANNDATS_ACT range: None to None


In [2]:
import csv

with open(r"C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL\data\ibes_quarterly_report\ibes_data.csv",
          encoding="utf-8", errors="replace") as f:
    reader = csv.reader(f)
    header = next(reader)
    print(header)
    for i, row in enumerate(reader):
        print(row)
        if i >= 4:
            break

['TICKER', 'CUSIP', 'OFTIC', 'CNAME', 'STATPERS', 'MEASURE', 'FISCALP', 'FPI', 'ESTFLAG', 'CURCODE', 'NUMEST', 'NUMUP', 'NUMDOWN', 'MEDEST', 'MEANEST', 'STDEV', 'HIGHEST', 'LOWEST', 'USFIRM', 'FPEDATS', 'ACTUAL', 'ANNDATS_ACT', 'ANNTIMS_ACT', 'CURR_ACT']
['0000', '87482X10', 'TLMR', 'TALMER BANCORP', '2014-04-17', 'EPS', 'QTR', '6', 'P', 'USD', '4', '0', '4', '0.07', '0.08', '0.01', '0.1', '0.07', '1', '2014-03-31', '0.12', '2014-05-06', '10:45:00', 'USD']
['0000', '87482X10', 'TLMR', 'TALMER BANCORP', '2014-05-15', 'EPS', 'QTR', '6', 'P', 'USD', '5', '3', '0', '0.13', '0.13', '0.01', '0.15', '0.12', '1', '2014-06-30', '0.27', '2014-08-06', '17:05:00', 'USD']
['0000', '87482X10', 'TLMR', 'TALMER BANCORP', '2014-06-19', 'EPS', 'QTR', '6', 'P', 'USD', '5', '0', '0', '0.13', '0.13', '0.01', '0.15', '0.12', '1', '2014-06-30', '0.27', '2014-08-06', '17:05:00', 'USD']
['0000', '87482X10', 'TLMR', 'TALMER BANCORP', '2014-07-17', 'EPS', 'QTR', '6', 'P', 'USD', '5', '0', '0', '0.13', '0.13', '0

In [1]:
import polars as pl
df = pl.read_parquet(r"C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL\data\ibes_quarterly_report\ibes_clean.parquet")
print("STATPERS range:", df["STATPERS"].min(), "to", df["STATPERS"].max())
print("ANNDATS_ACT range:", df["ANNDATS_ACT"].min(), "to", df["ANNDATS_ACT"].max())

STATPERS range: 2010-01-14 to 2026-02-19
ANNDATS_ACT range: 2005-04-26 to 2026-02-19


In [6]:
import sys
sys.path.append("src")
import polars as pl
from paths import IBES_DIR, PARQUET_DIR
from build_dispersion_events import get_cboe_date_bounds

def find_src_dir(start: Path = None) -> Path:
    """Walk upward from the current directory untill and'src' folder is found."""
    current = start or Path.cwd()
    for _ in range(5):
        candidate = current / "src"
        if candidate.exists():
            return candidate
        current = current.parent
    raise FileNotFoundError ("Could not locate a 'src' folder above current directory.")

SRC_DIR = find_src_dir()
sys.path.append(str(SRC_DIR))
from paths import IBES_DIR, PARQUET_DIR
from build_dispersion_events import get_cboe_date_bounds
    
# === 1. Which side of the MEANEST bound is actually responsible? ===
lo, hi = get_cboe_date_bounds(PARQUET_DIR)
df = pl.scan_parquet(IBES_DIR / "ibes_clean.parquet")
scoped = df.filter(
    (pl.col("MEASURE") == "EPS") & (pl.col("FISCALP") == "QTR")
    & pl.col("ANNDATS_ACT").is_not_null()
    & (pl.col("ANNDATS_ACT") >= lo) & (pl.col("ANNDATS_ACT") <= hi)
)
events = (
    scoped.filter(pl.col("STATPERS") < pl.col("ANNDATS_ACT"))
    .sort("STATPERS")
    .group_by(["OFTIC", "FPEDATS"], maintain_order=True)
    .agg(pl.all().last())
    .collect()
    .filter(pl.col("NUMEST") >= 2)
)
below_floor = events.filter(pl.col("MEANEST").is_not_null() & (pl.col("MEANEST").abs() <= 0.05)).height
above_ceiling = events.filter(pl.col("MEANEST").is_not_null() & (pl.col("MEANEST").abs() >= 10_000)).height
null_meanest = events.filter(pl.col("MEANEST").is_null()).height

print(f"Total before MEANEST bound: {events.height:,}")
print(f"Dropped -- MEANEST is null: {null_meanest:,}")
print(f"Dropped -- |MEANEST| <= 0.05 (floor): {below_floor:,}")
print(f"Dropped -- |MEANEST| >= 10,000 (ceiling): {above_ceiling:,}")

# === 2. Final top-20 check, nulls handled correctly this time ===
final = pl.read_parquet(IBES_DIR / "dispersion_events.parquet")
print("\n=== Top 20 in final cleaned file ===")
print(
    final.sort("dispersion_scaled", descending=True, nulls_last=True)
    .select(["OFTIC", "CNAME", "STATPERS", "STDEV", "MEANEST", "dispersion_scaled", "NUMEST"])
    .head(20)
           )

# === 3. Middle-of-distribution spot check === 
meadian = final["dispersion_scaled"].median()
print(f"\n=== Random sample near the median ({median:.4f}) ===")
print(
   final.filter((pl.col("dispersion_scaled") > median * 0.8) & (pl.col("dispersion_scaled") < median * 1.2))
    .select(["OFTIC", "CNAME", "STATPERS", "STDEV", "MEANEST", "dispersion_scaled", "NUMEST"])
    .sample(10, seed=1)
)

Total before MEANEST bound: 164,878
Dropped -- MEANEST is null: 34
Dropped -- |MEANEST| <= 0.05 (floor): 14,711
Dropped -- |MEANEST| >= 10,000 (ceiling): 541

=== Top 20 in final cleaned file ===
shape: (20, 7)
┌───────┬─────────────────┬────────────┬───────┬─────────┬───────────────────┬────────┐
│ OFTIC ┆ CNAME           ┆ STATPERS   ┆ STDEV ┆ MEANEST ┆ dispersion_scaled ┆ NUMEST │
│ ---   ┆ ---             ┆ ---        ┆ ---   ┆ ---     ┆ ---               ┆ ---    │
│ str   ┆ str             ┆ date       ┆ f64   ┆ f64     ┆ f64               ┆ i32    │
╞═══════╪═════════════════╪════════════╪═══════╪═════════╪═══════════════════╪════════╡
│ AA    ┆ ALCOA INC.      ┆ 2012-09-20 ┆ 0.18  ┆ 0.07    ┆ 1.85              ┆ 17     │
│ ABKFQ ┆ AMBAC FINANCIAL ┆ 2013-06-20 ┆ 2.73  ┆ 0.11    ┆ 1.85              ┆ 2      │
│ ABKFQ ┆ AMBAC FINANCIAL ┆ 2013-06-20 ┆ 2.37  ┆ 0.38    ┆ 1.85              ┆ 2      │
│ ABKFQ ┆ AMBAC FINANCIAL ┆ 2013-06-20 ┆ 2.4   ┆ 0.32    ┆ 1.85              ┆ 2     